# Capítulo 4.5 — Integração das previsões com a precificação de ativos

Notebook de implementação da seção 4.5 do TCC. Recebe os arquivos do experimento da seção 4.3 e produz:

1. Três trajetórias de CDI capitalizadas no horizonte de 22 dias úteis (previsão XGBoost, taxa implícita da DI, CDI realizado).
2. Valores futuros dos dois instrumentos representativos definidos em 3.11 (debênture pós-fixada CDI + spread fixo, perna pós-fixada de swap DI), sob cada uma das três curvas.
3. Erros de precificação, métricas agregadas, decomposição por sub-período (espelhando a Tabela 9 da 4.3) e DV01 (choque paralelo de 1 bp).
4. Figuras 17 e 18 e tabelas 12, 13 e 14 prontas para a seção.

**Parâmetros fixos do exercício** (definidos em 4.5.1):

- Nocional: **R$ 1.000.000,00**
- Prazo: **22 dias úteis** (alinhado ao alvo do modelo)
- Convenção: **252 dias úteis**
- Spread da debênture: **2,00 % a.a.** (representativo de emissões corporativas pós-fixadas)
- Calendário de feriados: **B3** (extraído implicitamente das datas da base diária consolidada)

**Inputs esperados** (faça upload no Colab antes de rodar a seção 2):

- `dados_modelo_tcc_cdi.xlsx` — aba `Diario_Consolidado`, usada para o calendário B3 e o CDI realizado.
- `dados_modelo_tcc_di_futuro.xlsx` — referência; o conteúdo já está consolidado no arquivo de resultados.
- `resultados_43_consolidado.xlsx` — previsões do XGBoost, taxas implícitas da DI e CDI realizado alinhados por t.

## 1. Imports e configuração estética

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Estilo gráfico (alinhado ao usado em 4.1-4.4)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi']      = 110
plt.rcParams['font.size']       = 10
plt.rcParams['axes.titlesize']  = 11
plt.rcParams['axes.labelsize']  = 10


## 2. Parâmetros do exercício

In [ ]:
NOCIONAL       = 1_000_000.0   # R$ 1 milhão
SPREAD_DEB_AA  = 0.02          # 2,00 % a.a. (debênture)
PRAZO_DU       = 22            # dias úteis (alinhado ao alvo do modelo)
BASE_DU        = 252           # convenção B3
CHOQUE_BP      = 1             # choque paralelo de 1 bp para DV01

# Fator de spread (constante: spread é fixo e idêntico nas três curvas)
SPREAD_FACTOR = (1 + SPREAD_DEB_AA) ** (PRAZO_DU / BASE_DU)
print(f'Fator de spread (22 du / 252): {SPREAD_FACTOR:.6f}')


## 3. Carregamento dos dados

**No Google Colab**, execute a célula abaixo e selecione os três arquivos `.xlsx` para upload. Em ambiente local, comente o bloco do Colab e ajuste `DATA_DIR` para o diretório que contém os arquivos.

In [ ]:
# Para uso no Google Colab, descomente as duas linhas abaixo:

DATA_DIR = Path('.')  # ajuste em ambiente local

arq_cdi = DATA_DIR / "dados/dados_modelo_tcc_cdi.xlsx"
arq_di  = DATA_DIR / 'dados/dados_modelo_tcc_di_futuro.xlsx'
arq_res = DATA_DIR / "dados/resultados_43_consolidado.xlsx"

# CDI diário oficial (para o calendário B3 e o CDI realizado)
df_cdi_raw = pd.read_excel(arq_cdi, sheet_name='Diario_Consolidado')
df_cdi = df_cdi_raw[['Data', 'CDI']].copy()
df_cdi['Data'] = pd.to_datetime(df_cdi['Data']).dt.normalize()
df_cdi = df_cdi.dropna().sort_values('Data').reset_index(drop=True)
df_cdi.columns = ['Data', 'CDI_diario_aa']

# DI futuro (referência; conteúdo já consolidado no arquivo de resultados)
df_di = pd.read_excel(arq_di, sheet_name='DI_Futuro')
df_di['Data'] = pd.to_datetime(df_di['Data']).dt.normalize()
df_di['Taxa_pct'] = df_di['Taxa_a.a.'] * 100  # decimal -> %

# Resultados 4.3 (insumo principal)
df_res = pd.read_excel(arq_res, sheet_name='Sheet1')
df_res['Data'] = pd.to_datetime(df_res['Data']).dt.normalize()
df_res = df_res.sort_values('Data').reset_index(drop=True)

print(f'CDI diário:    {len(df_cdi)} obs, de {df_cdi["Data"].min().date()} a {df_cdi["Data"].max().date()}')
print(f'DI futuro:     {len(df_di)} obs, contratos {df_di["Contrato"].unique().tolist()}')
print(f'Resultados 4.3:{len(df_res)} obs, de {df_res["Data"].min().date()} a {df_res["Data"].max().date()}')
df_res.head()


## 4. Calendário B3 e horizonte de 22 dias úteis

O calendário de dias úteis é extraído implicitamente das datas presentes em `df_cdi` (base diária publicada pela B3), evitando manter uma lista de feriados separada. Para cada data t do período de teste, identificamos a data de vencimento $T = t + 22$ du procurando a posição correspondente no calendário.

In [ ]:
calendario = df_cdi['Data'].reset_index(drop=True)
pos_de = {d: i for i, d in enumerate(calendario)}

def venc_22du(t):
    """Retorna T = t + 22 du, ou None se sair do calendário."""
    if t not in pos_de:
        return None
    novo_idx = pos_de[t] + PRAZO_DU
    if novo_idx >= len(calendario):
        return None
    return calendario.iloc[novo_idx]

df_res['Venc_22du'] = df_res['Data'].apply(venc_22du)

# Sanidade: t = 27/03/2026 deve ter T = 30/04/2026 (conforme 3.4.1)
print('Validação de horizonte:')
print(df_res[['Data','Venc_DI','Venc_22du']].head(3))
print(df_res[['Data','Venc_DI','Venc_22du']].tail(3))


## 5. Construção das três trajetórias e fatores de capitalização

Conforme decidido em 4.5.1, as três curvas são tratadas de forma simétrica:

- **XGBoost**: previsão pontual em $t$ interpretada como CDI constante ao longo dos 22 du seguintes.
- **DI implícita**: taxa implícita do contrato dominante em $t$ interpretada como CDI médio constante ao longo dos 22 du.
- **CDI realizado**: média realizada nos 22 du seguintes (alvo do modelo, em % a.a., já presente no arquivo de resultados).

O fator de capitalização para cada curva $X$ em $t$ é

$$FF_X(t) = \left(1 + \frac{i_X(t)}{100}\right)^{22/252}.$$

In [ ]:
def fator(i_aa_pct):
    """Fator de capitalização para PRAZO_DU dias úteis, taxa em % a.a."""
    return (1 + i_aa_pct / 100.0) ** (PRAZO_DU / BASE_DU)

df_res['FF_ML']    = fator(df_res['Previsao_ML'])
df_res['FF_DI']    = fator(df_res['DI_Implicita'])
df_res['FF_real']  = fator(df_res['CDI_realizado'])
df_res['FF_naive'] = fator(df_res['Previsao_Naive'])

df_res[['Data','Previsao_ML','DI_Implicita','CDI_realizado',
        'FF_ML','FF_DI','FF_real']].head()


## 6. Precificação dos dois instrumentos

Para cada $t$ do período de teste:

**Swap DI — perna pós-fixada** (sem spread):
$$\text{VF}_{\text{swap},X}(t) = N \cdot FF_X(t)$$

**Debênture CDI + spread fixo** ($s = 2{,}00$ % a.a.):
$$\text{VF}_{\text{deb},X}(t) = N \cdot FF_X(t) \cdot (1+s)^{22/252}$$

O fator de spread é constante e idêntico nas três curvas — não interfere na comparação relativa, conforme discutido em 3.11.

In [ ]:
for curva in ['ML','DI','real','naive']:
    df_res[f'VF_swap_{curva}'] = NOCIONAL * df_res[f'FF_{curva}']
    df_res[f'VF_deb_{curva}']  = df_res[f'VF_swap_{curva}'] * SPREAD_FACTOR

df_res[['Data',
        'VF_swap_ML','VF_swap_DI','VF_swap_real',
        'VF_deb_ML','VF_deb_DI','VF_deb_real']].head()


## 7. Erros de precificação

Erro de preço sob a curva $X$ é a diferença entre o valor futuro projetado por $X$ e o valor futuro sob CDI realizado:

$$\text{Erro}_X(t) = \text{VF}_X(t) - \text{VF}_{\text{real}}(t)$$

Valores positivos indicam que a curva $X$ **superestimou** o preço futuro do instrumento; negativos, subestimação.

In [ ]:
for inst in ['swap','deb']:
    for curva in ['ML','DI','naive']:
        df_res[f'erro_{inst}_{curva}'] = df_res[f'VF_{inst}_{curva}'] - df_res[f'VF_{inst}_real']

# Mantém apenas observações com horizonte completo (n=59)
mask = df_res['CDI_realizado'].notna()
df_eval = df_res[mask].copy()
print(f'Observações avaliáveis: {len(df_eval)}')
df_eval[['Data','erro_swap_ML','erro_swap_DI','erro_deb_ML','erro_deb_DI']].describe()


## 8. Tabela 12 — Métricas agregadas do erro de precificação

Métricas calculadas sobre as 59 observações com horizonte completo (02/01 a 27/03/2026):

- **MAE**: erro absoluto médio em R$
- **RMSE**: raiz do erro quadrático médio em R$
- **Viés**: erro médio com sinal preservado em R$
- **MAE relativo**: MAE como % do valor futuro realizado médio

In [ ]:
linhas = []
for inst, nome_inst in [('swap','Swap DI'),('deb','Debênture CDI + 2%')]:
    ref = df_eval[f'VF_{inst}_real']
    for curva, nome_curva in [('ML','XGBoost'),('DI','DI implícita'),('naive','Sem mudança')]:
        e = df_eval[f'erro_{inst}_{curva}']
        linhas.append({
            'Instrumento': nome_inst,
            'Curva': nome_curva,
            'MAE (R$)':       e.abs().mean(),
            'RMSE (R$)':      np.sqrt((e**2).mean()),
            'Viés (R$)':      e.mean(),
            'MAE (% do VF)':  e.abs().mean() / ref.mean() * 100,
        })
tabela_12 = pd.DataFrame(linhas)
print('Tabela 12 — Métricas agregadas (n=59)')
print(tabela_12.round(2).to_string(index=False))


## 9. Tabela 13 — Erro de precificação por sub-período

Decomposição em três sub-períodos delimitados pelas reuniões do Copom (espelhando a Tabela 9 de 4.3):

- **Pré-Copom de janeiro**: 02/01 a 27/01 (n=18)
- **Entre as duas reuniões**: 28/01 a 17/03 (n=33)
- **Pós-Copom de março**: 18/03 a 27/03 (n=8)

In [ ]:
copom1 = pd.Timestamp('2026-01-28')
copom2 = pd.Timestamp('2026-03-18')

def subperiodo(d):
    if d < copom1: return '1. Pré-Copom jan (02/01 a 27/01)'
    if d < copom2: return '2. Entre Copoms (28/01 a 17/03)'
    return '3. Pós-Copom mar (18/03 a 27/03)'

df_eval['Subperiodo'] = df_eval['Data'].apply(subperiodo)

for inst, nome_inst in [('swap','Swap DI'),('deb','Debênture')]:
    linhas = []
    for sp in sorted(df_eval['Subperiodo'].unique()):
        sub = df_eval[df_eval['Subperiodo']==sp]
        row = {'Sub-período': sp, 'n': len(sub)}
        for curva, nome_curva in [('ML','XGBoost'),('DI','DI'),('naive','Naive')]:
            row[f'MAE_{nome_curva}']  = sub[f'erro_{inst}_{curva}'].abs().mean()
            row[f'Viés_{nome_curva}'] = sub[f'erro_{inst}_{curva}'].mean()
        linhas.append(row)
    print(f'\n--- {nome_inst} ---')
    print(pd.DataFrame(linhas).round(2).to_string(index=False))


## 10. Tabela 14 — DV01 (choque paralelo de 1 bp)

DV01 é calculado como a variação no valor futuro projetado decorrente de um choque paralelo de +1 bp na taxa anualizada da curva. Para um instrumento bullet de 22 du sob convenção de 252 du:

$$DV01_X(t) = N \cdot \left[\left(1 + \frac{i_X(t)+0{,}01}{100}\right)^{22/252} - \left(1 + \frac{i_X(t)}{100}\right)^{22/252}\right] \cdot (1+s)^{22/252}$$

(o fator de spread é omitido no swap, que não tem spread.)

O DV01 varia apenas com o nível da taxa, por isso reportamos o valor médio sob cada curva no período de teste.

In [ ]:
CHOQUE_AA_PCT = CHOQUE_BP / 100.0   # 1 bp = 0,01 em % a.a.

def dv01_calc(i_aa_pct, com_spread):
    base = fator(i_aa_pct)
    choq = fator(i_aa_pct + CHOQUE_AA_PCT)
    sf = SPREAD_FACTOR if com_spread else 1.0
    return NOCIONAL * (choq - base) * sf

for col_taxa, nome in [('Previsao_ML','XGBoost'),
                       ('DI_Implicita','DI'),
                       ('CDI_realizado','Realizado')]:
    df_eval[f'DV01_swap_{nome}'] = dv01_calc(df_eval[col_taxa], False)
    df_eval[f'DV01_deb_{nome}']  = dv01_calc(df_eval[col_taxa], True)

dv01_resumo = pd.DataFrame({
    'Swap DI':    [df_eval['DV01_swap_XGBoost'].mean(),
                   df_eval['DV01_swap_DI'].mean(),
                   df_eval['DV01_swap_Realizado'].mean()],
    'Debênture':  [df_eval['DV01_deb_XGBoost'].mean(),
                   df_eval['DV01_deb_DI'].mean(),
                   df_eval['DV01_deb_Realizado'].mean()],
}, index=['Sob curva XGBoost','Sob curva DI','Sob CDI realizado'])

print('Tabela 14 — DV01 médio (R$ por 1 bp)')
print(dv01_resumo.round(4))


## 11. Figura 17 — Rentabilidade projetada dos dois instrumentos

Apresentamos a **rentabilidade projetada em T** (VF − N) em vez do valor absoluto, para que as diferenças entre as três curvas fiquem visíveis em escala interpretável.

In [ ]:
copom_dates = [pd.Timestamp('2026-01-28'), pd.Timestamp('2026-03-18')]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, inst, titulo in zip(axes, ['swap','deb'],
        ['Swap DI — perna pós-fixada','Debênture CDI + 2,00 % a.a.']):
    ax.plot(df_eval['Data'], df_eval[f'VF_{inst}_real']-NOCIONAL, 'k-', lw=2,   label='CDI realizado')
    ax.plot(df_eval['Data'], df_eval[f'VF_{inst}_ML']  -NOCIONAL, 'b-', lw=1.5, label='Previsão XGBoost')
    ax.plot(df_eval['Data'], df_eval[f'VF_{inst}_DI']  -NOCIONAL, 'r-', lw=1.5, label='DI implícita')
    for cd in copom_dates:
        ax.axvline(cd, color='gray', linestyle='--', alpha=0.7)
    ax.set_title(titulo)
    ax.set_ylabel('Rentabilidade projetada em T (R$)')
    ax.set_xlabel('Data')
    ax.legend(loc='best', fontsize=9)
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')

fig.suptitle('Figura 17 — Rentabilidade projetada dos dois instrumentos sob as três curvas (jan-mar/2026)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('figura_17_vf.png', dpi=200, bbox_inches='tight')
plt.show()


## 12. Figura 18 — Erros de precificação ao longo do teste

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, inst, titulo in zip(axes, ['swap','deb'],
        ['Swap DI — perna pós-fixada','Debênture CDI + 2,00 % a.a.']):
    ax.axhline(0, color='black', lw=0.8)
    ax.plot(df_eval['Data'], df_eval[f'erro_{inst}_ML'], 'b-', lw=1.5,
            marker='o', markersize=3, label='XGBoost − Realizado')
    ax.plot(df_eval['Data'], df_eval[f'erro_{inst}_DI'], 'r-', lw=1.5,
            marker='s', markersize=3, label='DI implícita − Realizado')
    for cd in copom_dates:
        ax.axvline(cd, color='gray', linestyle='--', alpha=0.7)
    ax.set_title(titulo)
    ax.set_ylabel('Erro de precificação (R$)')
    ax.set_xlabel('Data')
    ax.legend(loc='best', fontsize=9)

fig.suptitle('Figura 18 — Erro de precificação dos dois instrumentos (jan-mar/2026)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('figura_18_erros.png', dpi=200, bbox_inches='tight')
plt.show()


## 13. Export de resultados

Salva todos os outputs em um único arquivo `.xlsx` com múltiplas abas. As figuras já foram salvas como PNG nas células anteriores.

**No Colab**, após a execução, baixe o arquivo via aba lateral ou rodando `files.download('resultados_45_precificacao.xlsx')`.

In [ ]:
with pd.ExcelWriter('resultados_45_precificacao.xlsx', engine='openpyxl') as w:
    cols_export = ['Data','Previsao_ML','DI_Implicita','CDI_realizado','Previsao_Naive',
                   'Venc_22du','FF_ML','FF_DI','FF_real','FF_naive',
                   'VF_swap_ML','VF_swap_DI','VF_swap_real','VF_swap_naive',
                   'VF_deb_ML','VF_deb_DI','VF_deb_real','VF_deb_naive',
                   'erro_swap_ML','erro_swap_DI','erro_swap_naive',
                   'erro_deb_ML','erro_deb_DI','erro_deb_naive','Subperiodo']
    df_eval[cols_export].to_excel(w, sheet_name='Dados_diarios', index=False)
    tabela_12.to_excel(w, sheet_name='Tabela_12_agregada', index=False)

    for inst, nome_aba in [('swap','Tabela_13_Swap'), ('deb','Tabela_13_Debenture')]:
        linhas = []
        for sp in sorted(df_eval['Subperiodo'].unique()):
            sub = df_eval[df_eval['Subperiodo']==sp]
            row = {'Sub-periodo': sp, 'n': len(sub)}
            for curva, nome_curva in [('ML','XGBoost'),('DI','DI'),('naive','Naive')]:
                row[f'MAE_{nome_curva}']  = sub[f'erro_{inst}_{curva}'].abs().mean()
                row[f'Vies_{nome_curva}'] = sub[f'erro_{inst}_{curva}'].mean()
            linhas.append(row)
        pd.DataFrame(linhas).to_excel(w, sheet_name=nome_aba, index=False)

    dv01_resumo.to_excel(w, sheet_name='Tabela_14_DV01')

print('Arquivo salvo: resultados_45_precificacao.xlsx')
print('Figuras salvas: figura_17_vf.png, figura_18_erros.png')


---

**Saídas produzidas:**

- `resultados_45_precificacao.xlsx` — 5 abas: dados diários, Tabela 12 (agregada), Tabela 13 do swap, Tabela 13 da debênture, Tabela 14 (DV01).
- `figura_17_vf.png` — rentabilidade projetada dos dois instrumentos sob as três curvas.
- `figura_18_erros.png` — erros de precificação ao longo do trimestre.

Esses arquivos alimentam diretamente a redação da seção 4.5.